# DEIM D-FINE-Nano — VisDrone inference + mAP + small-object analysis (Kaggle)

Single-model run for the VisDrone detection/segmentation comparison.

**Kaggle setup (do this first):**
1. `Settings → Accelerator → GPU T4`, `Internet → ON` (needed for pip + checkpoint download).
2. **Add data:** `Add Input → Upload` your full VisDrone dataset (train + val). Expected layout:
```
/kaggle/input/<your-dataset>/train/images/*.jpg + train/annotations/*.txt
/kaggle/input/<your-dataset>/val/images/*.jpg   + val/annotations/*.txt   <- primary eval split
```
3. Run cells top to bottom. Outputs go to `/kaggle/working/results/` → use **Save Version** to persist.

**What this notebook does:** smoke-test → batch inference + timing → COCO mAP → small-object slice (APs/ARs) → gallery → export (`preds.json`, `metrics.json`, `timings.csv`).

**Known limitation (report this):** D-FINE-Nano is COCO-pretrained (80 classes); VisDrone has 10 drone-view classes. We dual-report (a) overlap-only mAP on 4 super-categories and (b) full-GT shift-penalized mAP — see Cell 9.

In [ ]:
# Cell 1 — environment check
import torch, platform, shutil
print('python ok | torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
!nvidia-smi -L 2>&1 | head -5
!free -g 2>&1 | head -3  # CPU RAM (Kaggle ~13-30GB)
print('platform:', platform.platform())

In [ ]:
# Cell 2 — install deps + importlib.resources patch check
# deimkit 0.2.1 used pkg_resources.resource_filename; repo patch swaps to importlib.resources.
# On Kaggle we apply the equivalent inline if needed (no repo files required).
!pip install -q deimkit pycocotools matplotlib pillow tqdm
import importlib.util, pathlib
spec = importlib.util.find_spec('deimkit.config')
print('deimkit.config at:', spec.origin if spec else 'NOT FOUND')
p = pathlib.Path(spec.origin)
t = p.read_text(encoding='utf-8')
print('needs patch (has pkg_resources):', 'pkg_resources' in t)
if 'pkg_resources' in t:
    t = t.replace('import pkg_resources', 'from importlib import resources as _resources')
    t = t.replace(
        'pkg_resources.resource_filename(\n                "deimkit", f"configs/{model_name}_coco.yml"\n            ),\n            pkg_resources.resource_filename(\n                "deimkit", f"configs/deim_dfine/{model_name}_coco.yml"\n            ),',
        'os.fspath(_resources.files("deimkit").joinpath(f"configs/{model_name}_coco.yml")),\n            os.fspath(_resources.files("deimkit").joinpath(f"configs/deim_dfine/{model_name}_coco.yml")),'
    )
    p.write_text(t, encoding='utf-8')
    print('patched inline OK')
else:
    print('already patched (or new version) — continuing')
from deimkit import list_models, load_model
print('models:', list_models())

In [ ]:
# Cell 3 — config + locate uploaded dataset
import os, glob
from pathlib import Path

SEED = 42
CONF_BASE = 0.05      # run inference once at low thresh; sweep 0.1/0.25/0.5 post-hoc
CONF_SWEEP = [0.1, 0.25, 0.5]
BATCH_SIZE = 16       # drop to 8/4 on OOM (large VisDrone frames)
MAX_IMAGES = None     # None = full val split; set 250 for a quick smoke run
SPLIT = 'val'         # primary eval split (no train-eval leakage)
OUT = Path('/kaggle/working/results'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'gallery').mkdir(exist_ok=True)

cands = sorted(Path('/kaggle/input').glob('*'))
print('kaggle inputs:', [c.name for c in cands])
ROOT = None
for c in cands:
    if ((c / 'val' / 'images').exists()) or ((c / 'train' / 'images').exists()):
        ROOT = c; break
    for sub in c.rglob('annotations'):
        if (sub.parent / 'images').exists():
            ROOT = sub.parent.parent; break
assert ROOT is not None, 'Dataset not found under /kaggle/input — upload it first (see header).'
print('dataset root:', ROOT)
IMG_DIR = (ROOT / SPLIT / 'images') if (ROOT / SPLIT / 'images').exists() else (ROOT / 'train' / 'images')
ANN_DIR = (ROOT / SPLIT / 'annotations') if (ROOT / SPLIT / 'annotations').exists() else (ROOT / 'train' / 'annotations')
if IMG_DIR != (ROOT / SPLIT / 'images'):
    print('WARNING: no val split found — falling back to train/ (pseudo-val; disclose in report)')
imgs = sorted(IMG_DIR.glob('*.jpg'))
print(f'{len(imgs)} images in {IMG_DIR}')

In [ ]:
# Cell 4 — parse VisDrone-DET txt -> COCO GT + EDA (class/area stats)
# VisDrone line: left,top,w,h,score,occlusion,truncation,category(0-11; keep 1-10)
import json, random
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt

VDRONE_NAMES = {1:'pedestrian',2:'people',3:'bicycle',4:'car',5:'van',6:'truck',7:'tricycle',8:'awning',9:'bus',10:'motor'}
random.seed(SEED)
sel = imgs if MAX_IMAGES is None else random.sample(imgs, min(MAX_IMAGES, len(imgs)))
print(f'evaluating on {len(sel)} images (seed={SEED})')

coco_gt = {'images': [], 'annotations': [], 'categories': [{'id':k,'name':v} for k,v in VDRONE_NAMES.items()]}
aid, areas, cls = 1, [], Counter()
img_info = {}
for i, ip in enumerate(sel):
    w, h = Image.open(ip).size
    img_info[ip.name] = {'id': i+1, 'w': w, 'h': h}
    coco_gt['images'].append({'id': i+1, 'file_name': ip.name, 'width': w, 'height': h})
    ap = ANN_DIR / (ip.stem + '.txt')
    if not ap.exists(): continue
    for line in ap.read_text().splitlines():
        parts = line.strip().split(',')
        if len(parts) < 8: continue
        x,y,bw,bh = map(float, parts[:4]); cat = int(parts[7])
        if cat not in VDRONE_NAMES or bw <= 0 or bh <= 0: continue
        x = min(max(0,x), w-1); y = min(max(0,y), h-1); bw = min(bw, w-x); bh = min(bh, h-y)
        coco_gt['annotations'].append({'id': aid, 'image_id': i+1, 'category_id': cat,
            'bbox': [x,y,bw,bh], 'area': bw*bh, 'iscrowd': 0})
        aid += 1; areas.append(bw*bh); cls[cat] += 1
(OUT / 'gt_coco.json').write_text(json.dumps(coco_gt))
print('gt boxes:', len(coco_gt['annotations']))
print('per-class:', {VDRONE_NAMES[k]: v for k,v in sorted(cls.items())})
import numpy as np
areas = np.array(areas)
print(f'small (<32^2=1024px): {(areas<1024).mean()*100:.1f}% | median area: {np.median(areas):.0f}px')
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].bar([VDRONE_NAMES[k] for k in sorted(cls)], [cls[k] for k in sorted(cls)]); ax[0].set_title('GT class histogram'); ax[0].tick_params(axis='x', rotation=45)
ax[1].hist(np.log10(areas+1), bins=50); ax[1].axvline(np.log10(1024), color='r', ls='--', label='small<1024px'); ax[1].set_title('log10 box area'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT/'eda_gt.png', dpi=120); plt.show()

In [ ]:
# Cell 5 — load D-FINE-Nano (dfine name first, deim fallback)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
predictor, used_name = None, None
for name in ['dfine_hgnetv2_n', 'deim_hgnetv2_n']:
    try:
        predictor = load_model(name, device='auto')
        used_name = name; print('loaded:', name); break
    except Exception as e:
        print(f'{name} failed: {type(e).__name__}: {str(e)[:200]}')
assert predictor is not None, 'Both model names failed — check internet/checkpoint cache.'
print('coco labels head:', (predictor.class_names[:5] if predictor.class_names else None))
try:
    from deimkit.model import get_model_checkpoint
    ckpt = get_model_checkpoint(used_name, None); import os
    print('checkpoint:', ckpt, f"({os.path.getsize(ckpt)/1e6:.1f} MB)" if os.path.exists(ckpt) else '')
except Exception as e:
    print('ckpt size lookup skipped:', str(e)[:150])
try:
    n = sum(p.numel() for p in predictor.model.parameters()); print(f'params: {n/1e6:.2f} M')
except Exception as e:
    print('param count skipped:', str(e)[:150])

In [ ]:
# Cell 6 — smoke test: single image + visualization
import matplotlib.pyplot as plt
demo = str(sel[0])
r = predictor.predict(demo, conf_threshold=0.25, visualize=True)
print('boxes:', r.boxes.shape, '| labels:', r.labels[:10], '| scores:', r.scores[:10] if len(r.scores) else r.scores)
print('boxes are xyxy absolute pixels:', r.boxes[:3] if len(r.boxes) else r.boxes)
if r.visualization is not None:
    plt.figure(figsize=(10,6)); plt.imshow(r.visualization); plt.axis('off'); plt.title(f'{sel[0].name} — {used_name} @0.25')
    plt.savefig(OUT/'gallery'/'smoke_demo.jpg', dpi=120, bbox_inches='tight'); plt.show()
else:
    print('no visualization returned; boxes printed above')

In [ ]:
# Cell 7 — batch inference (full subset) + timing  |  boxes xyxy->xywh, COCO ids kept raw here
import time, json, csv
import numpy as np
paths = [str(p) for p in sel]
t0 = time.perf_counter()
results = predictor.predict_batch(paths, conf_threshold=CONF_BASE, batch_size=BATCH_SIZE)
wall = time.perf_counter() - t0
assert len(results) == len(paths)
print(f'inference wall: {wall:.1f}s for {len(paths)} imgs -> {len(paths)/wall:.1f} FPS (incl. python overhead)')
# stable per-image latency on 30-sample subset
probe = paths[:30]
ts = []
with __import__('torch').no_grad():
    for _ in range(2):
        for pp in probe[:5]: predictor.predict(pp, conf_threshold=CONF_BASE)  # warmup
    for pp in probe:
        a = time.perf_counter(); predictor.predict(pp, conf_threshold=CONF_BASE); ts.append((time.perf_counter()-a)*1000)
ts = np.array(ts)
print(f'per-image latency: mean {ts.mean():.0f} ms | p50 {np.median(ts):.0f} | p95 {np.percentile(ts,95):.0f}')
raw = []  # (img_id, xyxy, coco_label, score)
for (ip, res) in zip(sel, results):
    iid = img_info[ip.name]['id']
    for b, l, s in zip(res.boxes, res.labels, res.scores):
        raw.append((iid, [float(b[0]), float(b[1]), float(b[2]), float(b[3])], int(l), float(s)))
print('raw detections @%.2f:' % CONF_BASE, len(raw))
json.dump({'model': used_name, 'conf_base': CONF_BASE, 'wall_s': wall, 'fps': len(paths)/wall,
             'lat_ms_mean': float(ts.mean()), 'lat_ms_p95': float(np.percentile(ts,95))},
            open(OUT/'timings.json','w'), indent=2)
with open(OUT/'timings.csv','w', newline='') as f:
    w = csv.writer(f); w.writerow(['model','n_images','wall_s','fps','lat_ms_mean','lat_ms_p95'])
    w.writerow([used_name, len(paths), f'{wall:.1f}', f"{len(paths)/wall:.1f}", f'{ts.mean():.0f}', f"{np.percentile(ts,95):.0f}"])
print('saved timings.json/csv')

In [ ]:
# Cell 8 — mAP (pycocotools) + conf sweep + dual reporting (overlap vs full shift-penalized)
# Mapping: VisDrone->super {1,2:PERSON=1; 4,5:CAR=2; 6:TRUCK=3; 9:BUS=4}; COCO {person0:1, car2:2, truck7:3, bus5:4}
# (a) OVERLAP: GT+preds filtered to 4 super-cats (fair number).  (b) FULL: all GT kept (unmapped=auto-miss).
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np

V2S = {1:1, 2:1, 4:2, 5:2, 6:3, 9:4}
C2S = {0:1, 2:2, 7:3, 5:4}
SUPER = {1:'person', 2:'car', 3:'truck', 4:'bus'}

def build_gt(mode):
    g = {'images': coco_gt['images'], 'annotations': [], 'categories': [{'id':k,'name':v} for k,v in SUPER.items()]}
    extra = {}
    for a in coco_gt['annotations']:
        if a['category_id'] in V2S:
            g['annotations'].append({**a, 'category_id': V2S[a['category_id']]})
        elif mode == 'full':
            nk = 10 + a['category_id']  # unique unmapped ids -> automatic misses
            extra[nk] = f"vdrone_{a['category_id']}"
            g['annotations'].append({**a, 'category_id': nk})
    for k, v in extra.items(): g['categories'].append({'id': k, 'name': v})
    return g

def build_dt(thr):
    d = []
    for (iid, xyxy, cl, s) in raw:
        if s < thr or cl not in C2S: continue
        x1,y1,x2,y2 = xyxy; d.append({'image_id': iid, 'category_id': C2S[cl],
            'bbox': [x1, y1, max(0,x2-x1), max(0,y2-y1)], 'score': s})
    return d

def run_eval(gt_dict, dt):
    import tempfile, os
    with tempfile.TemporaryDirectory() as td:
        gp, dp = os.path.join(td,'g.json'), os.path.join(td,'d.json')
        json.dump(gt_dict, open(gp,'w')); json.dump(dt, open(dp,'w'))
        coco = COCO(gp); coco.loadRes(dp)
        ev = COCOeval(coco, coco, 'bbox'); ev.evaluate(); ev.accumulate(); ev.summarize()
        return ev.stats

rows, metrics = {}, {}
for mode in ['overlap', 'full']:
    gt = build_gt(mode)
    for thr in CONF_SWEEP:
        dt = build_dt(thr)
        if not dt: print(f'{mode} @{thr}: no detections'); continue
        s = run_eval(gt, dt)
        rows[(mode, thr)] = {'mAP': round(float(s[0]),4), 'mAP50': round(float(s[1]),4), 'mAP75': round(float(s[2]),4),
            'APs': round(float(s[3]),4), 'APm': round(float(s[4]),4), 'APl': round(float(s[5]),4), 'n_pred': len(dt)}
        print(f"{mode:7s} conf={thr}: mAP={s[0]:.3f} mAP50={s[1]:.3f} APs={s[3]:.3f} APm={s[4]:.3f} APl={s[5]:.3f} (n={len(dt)})")
    metrics[mode] = {str(k[1]): v for k, v in rows.items() if k[0] == mode}
json.dump({'model': used_name, 'n_images': len(sel), **metrics}, open(OUT/'metrics.json','w'), indent=2)
print('saved metrics.json — PRIMARY for report: overlap @0.25 (fair) + full @0.25 (shift penalty)')

In [ ]:
# Cell 9 — small-object deep dive (recall vs area + relative-area variant)
import numpy as np, matplotlib.pyplot as plt
gt_areas = np.array([a['area'] for a in coco_gt['annotations']])
dt_by_thr = {}
for thr in CONF_SWEEP:
    ds = [(iid, (b[2]-b[0])*(b[3]-b[1]), s) for (iid, b, c, s) in raw if s >= thr and c in C2S]
    dt_by_thr[thr] = ds
    print(f'conf {thr}: {len(ds)} mapped preds (person/car/truck/bus only)')
bins = np.logspace(np.log10(100), np.log10(max(gt_areas.max(), 200000)), 12)
ct, _ = np.histogram(gt_areas, bins=bins)
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].hist(gt_areas, bins=60, label='GT'); ax[0].set_xscale('log'); ax[0].axvline(1024, color='r', ls='--', label='COCO small<1024')
ax[0].axvline(9216, color='orange', ls='--', label='medium<96^2'); ax[0].legend(); ax[0].set_title('GT box-area distribution')
frac = [(gt_areas < e).mean()*100 for e in [1024, 9216]]
ax[1].bar(['small<32²', 'small+med<96²'], frac); ax[1].set_ylabel('% of GT'); ax[1].set_title('Small-object dominance (VisDrone)')
plt.tight_layout(); plt.savefig(OUT/'small_object_areas.png', dpi=120); plt.show()
print('COCO-size APs/APm/APl are in metrics.json (Cell 8) — quote APs vs APm/APl gap in report.')
# relative-area variant (altitude-aware): area / image_area < 1%% = small regardless of px
rel = np.array([a['area']/(img_info[[k for k,v in img_info.items() if v['id']==a['image_id']][0]]['w']*img_info[[k for k,v in img_info.items() if v['id']==a['image_id']][0]]['h']) for a in coco_gt['annotations']])
print(f'relative-small (<1%% of frame): {(rel<0.01).mean()*100:.1f}% of GT')

In [ ]:
# Cell 10 — qualitative gallery (dense / night / occlusion hand-picks + randoms)
import matplotlib.pyplot as plt
from pathlib import Path
picks = list(sel[:4]) + list(__import__('random').Random(42).sample(sel, min(6, len(sel))))
for ip in picks:
    r = predictor.predict(str(ip), conf_threshold=0.25, visualize=True)
    img = r.visualization if r.visualization is not None else plt.imread(str(ip))
    plt.figure(figsize=(10,6)); plt.imshow(img); plt.axis('off')
    plt.title(f"{ip.name} — n={len(r.boxes)} @0.25")
    plt.savefig(OUT/'gallery'/f'{ip.stem}_pred.jpg', dpi=110, bbox_inches='tight'); plt.show()
print('gallery saved to', OUT/'gallery')

## Conclusions (fill after run) + export checklist

1. **Overall:** quote `metrics.json → overlap @0.25` mAP / mAP50 / APs-APm-APl gap.
2. **Small objects:** % small GT, APs vs APm/APl, recall-vs-area observation.
3. **Efficiency:** `timings.json` wall/FPS + per-image mean/p95; ckpt MB + params from Cell 5.
4. **Deployment:** T4 GPU numbers here vs expected CPU (~150–300 ms/img on i5-1135G7); real-time? YOLO/DEIM-Nano yes on GPU, no on CPU.
5. **Dataset shift:** overlap (fair) vs full (penalized) delta = cost of VisDrone-only classes (tricycle/awning/motor).

**Export:** `/kaggle/working/results/` contains `gt_coco.json`, `metrics.json`, `timings.json/csv`, `eda_gt.png`, `small_object_areas.png`, `gallery/*.jpg` → **Save Version** (or Download) and attach to report/notebook appendix.